In [0]:
spark


In [0]:
%sql

create schema bronze;

In [0]:
%sql 
create schema silver;

In [0]:
%sql 
create schema gold

In [0]:
%sql
SHOW STORAGE CREDENTIALS;

external tables for bronze layer

below code will read data from landing bucket and insert into bronze layer external tables. if any new file has been added to those landing buket same script will read based on the checkpoint and only load the new data into those tables.

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

landing_base = "s3://de-case-study-landing"
checkpoint_base = "s3://de-case-study-checkpoints"
bronze_base = "s3://de-case-study-bronze"

tables = {
    "orders": {
        "source_path": f"{landing_base}/orders/",
        "target_table": "bronze.orders_ext",
        "target_path": f"{bronze_base}/orders/",
        "checkpoint": f"{checkpoint_base}/bronze/orders_ext"
    },
    "customers": {
        "source_path": f"{landing_base}/customers/",
        "target_table": "bronze.customers_ext",
        "target_path": f"{bronze_base}/customers/",
        "checkpoint": f"{checkpoint_base}/bronze/customers_ext"
    },
    "order_items": {
        "source_path": f"{landing_base}/order_items/",
        "target_table": "bronze.order_items_ext",
        "target_path": f"{bronze_base}/order_items/",
        "checkpoint": f"{checkpoint_base}/bronze/order_items_ext"
    },
    "products": {
        "source_path": f"{landing_base}/products/",
        "target_table": "bronze.products_ext",
        "target_path": f"{bronze_base}/products/",
        "checkpoint": f"{checkpoint_base}/bronze/products_ext"
    }
}

for table_name, cfg in tables.items():

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferColumnTypes", "false")
        .option("cloudFiles.schemaLocation", f"{cfg['checkpoint']}/schema")
        .option("rescuedDataColumn", "_rescued_data")
        .load(cfg["source_path"])
    )

    bronze_df = (
        df.withColumn("source_system", lit("s3_landing"))
          .withColumn("ingestion_batch_id", lit("initial_load"))
          .withColumn("ingestion_timestamp", current_timestamp())
          .withColumn("source_file", col("_metadata.file_path"))
    )

    (
        bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{cfg['checkpoint']}/checkpoint")
        .option("path", cfg["target_path"])
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(cfg["target_table"])
    )

In [0]:
display(spark.table("de_case_study.bronze.products_ext"))